# Session 3 — Solutions

Worked solutions for signal region, cutflow, yields, and control regions (Z CR, Top CR). Uses one file per dataset from `config/datasets_2017_short.yaml`; SR plots are MC-only; data can be plotted in CRs.

## 3. Setup and cutflow (Ex 3.1, 3.3)


In [ ]:
# Ensure project root is on sys.path (for SWAN / any kernel)
import sys, os
sys.path.append("..")


import numpy as np
import awkward as ak
import matplotlib.pyplot as plt
import pandas as pd

import mplhep as hep

hep.style.use("CMS")

from config.datasets_2017 import get_short_datasets_meta, get_trigger_list
from processor.bbdm_processor import get_nanoevents, select_good_jets, count_bjets, n_tight_leptons, RECOIL_MIN, cos_theta_star, met_pf_calo_consistency_mask

# Normalization
# We scale MC yields to the target integrated luminosity.
# - 41.5/fb is the (approx.) 2017 dataset luminosity for this exercise.
# - xsec is in pb, so convert 41.5/fb → 41.5*1000 pb^{-1}.
LUMI_PB = 41.5 * 1000

# Load one file per dataset from config/datasets_2017_short.yaml.
# We cap to 10k events so the notebook runs quickly in an interactive session.
MAX_EVENTS = 10_000
datasets = get_short_datasets_meta()

events_by_dataset = {}
labels = {}
xsecs = {}
is_data = {}
norm_factors = {}

for name, meta in datasets.items():
    # Metadata for plotting and scaling
    labels[name] = meta.get("label", name)
    xsecs[name] = meta.get("xsec")
    is_data[name] = bool(meta.get("isData", False))

    # Load the NanoEvents (one ROOT file per dataset for the short exercise)
    path_list = meta.get("files", [])
    if path_list:
        ev = get_nanoevents(path_list[0])[:MAX_EVENTS]
        events_by_dataset[name] = ev

        # Simple normalization: weight every event by (xsec * lumi) / N_events.
        # This is *not* a full analysis weight model; it is enough for shape/yield practice.
        if not is_data[name]:
            xsec = xsecs.get(name)
            if xsec is None:
                norm_factors[name] = None
            else:
                norm_factors[name] = (float(xsec) * LUMI_PB) / float(len(ev))

# MC-only for SR: background names (exclude data and signal)
# In the SR we typically compare MC backgrounds (data is usually blinded / not used).
bkg_names = [k for k in events_by_dataset if (not is_data.get(k, False)) and ("bbDM" not in k)]

# Matplotlib legend labels (LaTeX)
latex_labels = {
    "DYJets": r"$Z(\ell\ell)+$jets",
    "ZJets": r"$Z(\nu\bar{\nu})+$jets",
    "WJets": r"$W(\ell\nu)+$jets",
    "DIBOSON": r"WW",
    "STop": r"Single $t$",
    "Top": r"$t\bar{t}$",
    "SMH": r"SMH",
    "bbDM_2HDMa_LO_5f": r"bbDM (2HDM+a)",
    "MET_Run2017B": r"Data",
}

# Sanity printout: xsec, N, normalization factor
for name in bkg_names:
    print(name, "xsec=", xsecs.get(name), "N=", len(events_by_dataset[name]), "norm=", norm_factors.get(name))

# Cutflow per dataset (Ex 3.1)
# We build the signal-region selection step-by-step and count how many events survive.
# This helps diagnose *which* cut is most powerful and whether selections behave as expected.

def run_cutflow(events):
    # Trigger OR (first cut)
    trigger_list = get_trigger_list()
    hlt_fields = set(events.HLT.fields) if hasattr(events, "HLT") and hasattr(events.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(events.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | events.HLT[tname]
    n0 = int(ak.sum(trigger_mask))
    events = events[trigger_mask]

    # Baseline objects
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)          # per-event jet multiplicity
    nlep = n_tight_leptons(events)     # per-event tight lepton multiplicity

    # Event-level MET
    met = events.MET.pt
    met_phi = events.MET.phi

    # Δφ(jet, MET): suppress QCD-like events where MET is aligned with a mismeasured jet.
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5

    # Require the two leading jets to be b-tagged (medium WP).
    # We pad to length 2 so events with <2 jets don't crash; missing entries become None.
    jets2 = ak.pad_none(good_jets, 2)
    lead2_b = (
        ak.fill_none(jets2[:, 0].btagDeepFlavB > 0.2783, False)
        & ak.fill_none(jets2[:, 1].btagDeepFlavB > 0.2783, False)
    )

    # Jet multiplicity window (analysis-style categorization).
    jets_2to3 = (njets >= 2) & (njets <= 3)

    # Jet-based preselection for the SR
    sr_jets = jets_2to3 & lead2_b

    # Cutflow counts (PF–calo after Δφ, before MET)
    n1 = ak.sum(sr_jets)
    n2 = ak.sum(sr_jets & (nlep == 0))
    n3 = ak.sum(sr_jets & (nlep == 0) & dphi_cut)
    met_pf_calo_ok = met_pf_calo_consistency_mask(events)
    n4 = ak.sum(sr_jets & (nlep == 0) & dphi_cut & met_pf_calo_ok)
    n5 = ak.sum(sr_jets & (nlep == 0) & dphi_cut & met_pf_calo_ok & (met > RECOIL_MIN))
    recoil_key = f"Recoil>{int(RECOIL_MIN)}"

    return {
        "trigger": n0,
        "presel": int(n1),
        "0lep": int(n2),
        "Δφ>0.5": int(n3),
        "|ΔMET(PF-calo)|<0.5": int(n4),
        recoil_key: int(n5),
    }


cutflow_by_dataset = {name: run_cutflow(ev) for name, ev in events_by_dataset.items()}
for name in cutflow_by_dataset:
    print(labels.get(name, name), cutflow_by_dataset[name])

# Yield table (Ex 3.3): one row per cut, one column per process
recoil_key = f"Recoil>{int(RECOIL_MIN)}"
cuts = ["Trigger (OR)", "2≤Njets≤3, leading 2 b-tag", "0 leptons", "Δφ>0.5", "|ΔMET(PF-calo)|<0.5 GeV", f"Recoil>{int(RECOIL_MIN)} GeV"]
keys = ["trigger", "presel", "0lep", "Δφ>0.5", "|ΔMET(PF-calo)|<0.5", recoil_key]
data = {"Cut": cuts}
for name in list(events_by_dataset.keys())[:6]:
    data[labels.get(name, name)] = [cutflow_by_dataset[name][k] for k in keys]
df = pd.DataFrame(data)
print(df)

## 4. MET in signal region — MC only (Ex 3.2)


In [ ]:
# MET in SR: MC-only stacked (no data in signal region), scaled to 41.5/fb
# We apply the *same SR mask* as in the cutflow and then plot MET for each background.
bins_met = np.linspace(250, 600, 50)
met_arrays, w_arrays, leg_labels = [], [], []

trigger_list = get_trigger_list()
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue

    ev = events_by_dataset[name]
    hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev.HLT[tname]
    ev = ev[trigger_mask]

    # Build SR selection ingredients
    good_jets = select_good_jets(ev)
    njets = ak.num(good_jets)
    nlep = n_tight_leptons(ev)
    met = ev.MET.pt

    met_phi = ev.MET.phi
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5

    jets2 = ak.pad_none(good_jets, 2)
    lead2_b = (
        ak.fill_none(jets2[:, 0].btagDeepFlavB > 0.2783, False)
        & ak.fill_none(jets2[:, 1].btagDeepFlavB > 0.2783, False)
    )

    # Final SR mask (PF–calo after Δφ, before MET)
    sr_mask = (
        (njets >= 2)
        & (njets <= 3)
        & lead2_b
        & (nlep == 0)
        & dphi_cut
        & met_pf_calo_consistency_mask(ev)
        & (met > RECOIL_MIN)
    )

    # Collect MET values passing SR and apply a flat normalization weight
    recoil = met[sr_mask]
    flat = ak.to_numpy(ak.ravel(recoil))
    if len(flat) == 0:
        continue

    met_arrays.append(flat)
    w_arrays.append(np.full_like(flat, norm, dtype=float))
    leg_labels.append(latex_labels.get(name, labels.get(name, name)))

if met_arrays:
    plt.hist(
        met_arrays,
        bins=bins_met,
        weights=w_arrays,
        label=leg_labels,
        stacked=True,
        histtype="stepfilled",
        alpha=0.7,
    )
plt.xlabel("MET [GeV]")
plt.ylabel("Events (scaled)")
plt.title("MET in signal region (MC only)")
plt.legend()
plt.show()

## 5. cos θ* distribution in the signal region

Main angular observable: **$\cos \theta^* = \left| \tanh\left(\frac{\Delta \eta}{2}\right) \right| \quad \text{with } \Delta \eta = \eta_{\text{jet0}} - \eta_{\text{jet1}}$** (two leading jets). Same SR selection as MET; MC-only, scaled to $41.5\,\mathrm{fb}^{-1}$.


In [ ]:
# cos(theta*) in SR (main observable)
# cos(theta*) is the primary angular observable for this exercise:
#   cos θ* = |tanh(Δη/2)| with Δη = η(jet0) − η(jet1)
# We compute it from the two leading jets *after* SR selection.
trigger_list = get_trigger_list()
bins_cts = np.linspace(0, 1, 25)
cts_arrays, cts_w, cts_labels = [], [], []
for name in bkg_names:
    if name not in events_by_dataset:
        continue
    norm = norm_factors.get(name)
    if norm is None:
        continue
    ev = events_by_dataset[name]
    hlt_fields = set(ev.HLT.fields) if hasattr(ev, "HLT") and hasattr(ev.HLT, "fields") else set()
    trigger_mask = ak.zeros_like(ev.event, dtype=bool)
    for tname in trigger_list:
        if tname in hlt_fields:
            trigger_mask = trigger_mask | ev.HLT[tname]
    ev = ev[trigger_mask]
    good_jets = select_good_jets(ev)
    njets = ak.num(good_jets)
    nlep = n_tight_leptons(ev)
    met = ev.MET.pt
    met_phi = ev.MET.phi
    dphi = np.abs(good_jets.phi - met_phi)
    dphi = ak.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    min_dphi = ak.min(dphi, axis=1)
    dphi_cut = min_dphi > 0.5
    # Leading-2 b-tag requirement: pad to 2 jets so indexing is safe.
    jets2 = ak.pad_none(good_jets, 2)
    lead2_b = ak.fill_none(jets2[:, 0].btagDeepFlavB > 0.2783, False) & ak.fill_none(jets2[:, 1].btagDeepFlavB > 0.2783, False)

    # Full SR mask (PF–calo after Δφ, before MET)
    sr_mask = (
        (njets >= 2)
        & (njets <= 3)
        & lead2_b
        & (nlep == 0)
        & dphi_cut
        & met_pf_calo_consistency_mask(ev)
        & (met > RECOIL_MIN)
    )

    # Two leading jets in SR to define Δη and hence cos(theta*)
    good_jets_sr = good_jets[sr_mask]
    has_two = ak.num(good_jets_sr) >= 2
    jets_pad = ak.pad_none(good_jets_sr, 2)
    jet0 = jets_pad[:, 0]
    jet1 = jets_pad[:, 1]

    # Only compute for events where jet1 exists (non-None)
    mask = has_two & ~ak.is_none(jet1)
    if ak.sum(mask) == 0:
        continue
    cts = cos_theta_star(jet0[mask], jet1[mask])
    flat = ak.to_numpy(ak.ravel(cts))
    if len(flat) == 0:
        continue
    cts_arrays.append(flat)
    cts_w.append(np.full_like(flat, norm, dtype=float))
    cts_labels.append(latex_labels.get(name, labels.get(name, name)))
if cts_arrays:
    plt.figure()
    plt.hist(cts_arrays, bins=bins_cts, weights=cts_w, label=cts_labels, stacked=True, histtype="stepfilled", alpha=0.7)
    plt.xlabel("$\cos(\Theta)$*")
    plt.ylabel("Events (scaled)")
    plt.title("$\cos(\Theta)$* in signal region (MC only)")
    plt.legend()
    plt.show()

## 7. Z control region (dilepton) — data and MC

Preselection: 
- $\geq 1$ jet, leading jet $p_{\mathrm{T}} > 100~\mathrm{GeV}$

- $\Delta\phi(\text{jet}, \mathrm{MET}) > 0.5$

- $\mathrm{PF\text{-}calo~MET} < 0.5~\mathrm{GeV}$

- Recoil $> 250~\mathrm{GeV}$  
    - $\left(U = -\left(\vec{p}_{\mathrm{T}}^{\,\text{miss}} + \sum \vec{p}_{\mathrm{T}}^{\,\ell}\right)\right)$

- Exactly 2 OSSF leptons (ee or $\mu\mu$)

- Leading lepton $p_{\mathrm{T}} > 30~\mathrm{GeV}$

- $70 < m_{\ell\ell} < 110~\mathrm{GeV}$

In [ ]:
from processor.bbdm_processor import select_tight_electrons, select_tight_muons, recoil_pt, met_pf_calo_consistency_mask

PRESEL_RECOIL_MIN = 250.0
LEAD_JET_PT_MIN = 100.0
Z_CR_MLL_LO, Z_CR_MLL_HI = 70.0, 110.0
LEAD_LEP_PT_CR = 30.0

def z_cr_mask(events):
    """Z→ℓℓ control region mask.

    Purpose: select a clean Z(ee/μμ) sample to validate modeling of Z+jets
    (and extrapolate to Z→νν backgrounds in SR-style analyses).

    Selection (simplified for this exercise):
    - **Preselection**: recoil > 250 GeV, ≥1 jet, leading jet pT > 100 GeV
    - **Z candidate**: exactly 2 OSSF leptons (ee or μμ), leading lepton pT > 30 GeV
    - **Mass window**: 70 < m_ll < 110 GeV
    """
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)

    met_pt, met_phi = events.MET.pt, events.MET.phi

    # Tight leptons are defined in the processor module (Session 2 selections)
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)

    # Exactly two leptons of the same flavor and opposite sign (OSSF)
    nele, nmu = ak.count(tight_ele.pt, axis=1), ak.count(tight_mu.pt, axis=1)
    two_ee = (nele == 2) & (nmu == 0) & (ak.sum(tight_ele.charge, axis=1) == 0)
    two_mumu = (nele == 0) & (nmu == 2) & (ak.sum(tight_mu.charge, axis=1) == 0)

    # Build dilepton 4-vectors (pad so [:,0] and [:,1] exist)
    tight_ele_pad = ak.pad_none(tight_ele, 2)
    tight_mu_pad = ak.pad_none(tight_mu, 2)
    pair_ee = tight_ele_pad[:, 0] + tight_ele_pad[:, 1]
    pair_mumu = tight_mu_pad[:, 0] + tight_mu_pad[:, 1]

    # Recoil is U = −(pTmiss + Σ pT^lep). We construct Σ pT^lep components.
    sum_lep_px = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.cos(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.cos(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    sum_lep_py = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.sin(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.sin(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)
    dphi_j = np.abs(good_jets.phi - met_phi)
    dphi_j = ak.where(dphi_j > np.pi, 2 * np.pi - dphi_j, dphi_j)
    min_dphi = ak.min(dphi_j, axis=1)
    dphi_cut = min_dphi > 0.5
    met_pf_calo_ok = met_pf_calo_consistency_mask(events)
    presel = (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN) & dphi_cut & met_pf_calo_ok & (recoil > PRESEL_RECOIL_MIN)
    mll_ee = ak.where(two_ee, ak.fill_none(pair_ee.mass, -1.0), ak.full_like(met_pt, -1.0))
    mll_mumu = ak.where(two_mumu, ak.fill_none(pair_mumu.mass, -1.0), ak.full_like(met_pt, -1.0))
    mll = ak.where(two_ee, mll_ee, ak.where(two_mumu, mll_mumu, ak.full_like(met_pt, -1.0)))
    lead_lep_pt = ak.where(two_ee, ak.max(tight_ele.pt, axis=1), ak.where(two_mumu, ak.max(tight_mu.pt, axis=1), ak.full_like(met_pt, 0.0)))
    return presel & (two_ee | two_mumu) & (lead_lep_pt > LEAD_LEP_PT_CR) & (mll > Z_CR_MLL_LO) & (mll < Z_CR_MLL_HI)

def z_cr_recoil(events):
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele, tight_mu = select_tight_electrons(events), select_tight_muons(events)
    nele, nmu = ak.count(tight_ele.pt, axis=1), ak.count(tight_mu.pt, axis=1)
    two_ee = (nele == 2) & (nmu == 0) & (ak.sum(tight_ele.charge, axis=1) == 0)
    two_mumu = (nele == 0) & (nmu == 2) & (ak.sum(tight_mu.charge, axis=1) == 0)
    tight_ele_pad = ak.pad_none(tight_ele, 2)
    tight_mu_pad = ak.pad_none(tight_mu, 2)
    pair_ee = tight_ele_pad[:, 0] + tight_ele_pad[:, 1]
    pair_mumu = tight_mu_pad[:, 0] + tight_mu_pad[:, 1]
    sum_lep_px = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.cos(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.cos(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    sum_lep_py = ak.where(two_ee, ak.fill_none(pair_ee.pt, 0.0) * np.sin(ak.fill_none(pair_ee.phi, 0.0)), ak.where(two_mumu, ak.fill_none(pair_mumu.pt, 0.0) * np.sin(ak.fill_none(pair_mumu.phi, 0.0)), ak.full_like(met_pt, 0.0)))
    return recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)

fig, ax = plt.subplots()
for name in list(events_by_dataset.keys())[:5]:
    ev = events_by_dataset[name]
    mask = z_cr_mask(ev)
    recoil = z_cr_recoil(ev)[mask]
    if len(ak.ravel(recoil)) == 0:
        continue
    if is_data.get(name, False):
        w = None
    else:
        norm = norm_factors.get(name)
        if norm is None:
            continue
        w = np.full_like(ak.to_numpy(ak.ravel(recoil)), norm, dtype=float)
    ax.hist(ak.to_numpy(ak.ravel(recoil)), bins=30, range=(250, 600), weights=w, label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax.set_xlabel("Recoil [GeV]")
ax.set_ylabel("Events")
ax.set_title("Z CR (dilepton): Recoil (data and MC)")
ax.legend()
plt.show()


In [ ]:
# cos(theta*) in Z CR (data and MC)
fig2, ax2 = plt.subplots()
for name in list(events_by_dataset.keys()):
    ev = events_by_dataset[name]
    mask = z_cr_mask(ev)
    good_jets_cr = select_good_jets(ev)[mask]
    has_two = ak.num(good_jets_cr) >= 2
    jets_pad = ak.pad_none(good_jets_cr, 2)
    jet0 = jets_pad[:, 0]
    jet1 = jets_pad[:, 1]
    mask2 = has_two & ~ak.is_none(jet1)
    if ak.sum(mask2) == 0:
        continue
    cts = cos_theta_star(jet0[mask2], jet1[mask2])
    flat = ak.to_numpy(ak.ravel(cts))
    if is_data.get(name, False):
        w = None
    else:
        norm = norm_factors.get(name)
        if norm is None:
            continue
        w = np.full_like(flat, norm, dtype=float)
    ax2.hist(flat, bins=25, range=(0, 1), weights=w, label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax2.set_xlabel("$\cos(\Theta)$*")
ax2.set_ylabel("Events")
ax2.set_title("Z CR (dilepton): $\cos(\Theta)$* (data and MC)")
ax2.legend()
plt.show()

## 7. Top control region (single-lepton) — data and MC

- Exactly one lepton with $p_{\mathrm{T}} > 30~\mathrm{GeV}$

- $m_{\mathrm{T}} < 160~\mathrm{GeV}$

- $\geq 2$ b-jets

- $\geq 2$ non-b jets

In [ ]:
from processor.bbdm_processor import met_pf_calo_consistency_mask

TOP_CR_MT_MAX = 160.0

def top_cr_mask(events):
    good_jets = select_good_jets(events)
    njets = ak.num(good_jets)
    nbjets = count_bjets(good_jets)
    lead_jet_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele = select_tight_electrons(events)
    tight_mu = select_tight_muons(events)
    one_ele = (ak.count(tight_ele.pt, axis=1) == 1) & (ak.count(tight_mu.pt, axis=1) == 0)
    one_mu = (ak.count(tight_ele.pt, axis=1) == 0) & (ak.count(tight_mu.pt, axis=1) == 1)
    lep_pt = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.pt), ak.where(one_mu, ak.firsts(tight_mu.pt), ak.full_like(met_pt, 0.0))), 0.0)
    lep_phi = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.phi), ak.where(one_mu, ak.firsts(tight_mu.phi), ak.full_like(met_pt, 0.0))), 0.0)
    sum_lep_px = lep_pt * np.cos(lep_phi)
    sum_lep_py = lep_pt * np.sin(lep_phi)
    recoil = recoil_pt(met_pt, met_phi, sum_lep_px, sum_lep_py)
    dphi_j = np.abs(good_jets.phi - met_phi)
    dphi_j = ak.where(dphi_j > np.pi, 2 * np.pi - dphi_j, dphi_j)
    min_dphi = ak.min(dphi_j, axis=1)
    dphi_cut = min_dphi > 0.5
    met_pf_calo_ok = met_pf_calo_consistency_mask(events)
    presel = (njets >= 1) & (lead_jet_pt > LEAD_JET_PT_MIN) & dphi_cut & met_pf_calo_ok & (recoil > PRESEL_RECOIL_MIN)
    dphi = np.abs(ak.to_numpy(met_phi) - ak.to_numpy(lep_phi))
    dphi = np.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    mt = np.sqrt(2.0 * ak.to_numpy(met_pt) * ak.to_numpy(lep_pt) * (1.0 - np.cos(dphi)))
    mt = ak.Array(mt)
    n_non_b = njets - nbjets
    return presel & (n_tight_leptons(events) == 1) & (lep_pt > LEAD_LEP_PT_CR) & (mt < TOP_CR_MT_MAX) & (nbjets >= 2) & (n_non_b >= 2)

def top_cr_recoil(events):
    met_pt, met_phi = events.MET.pt, events.MET.phi
    tight_ele, tight_mu = select_tight_electrons(events), select_tight_muons(events)
    one_ele = (ak.count(tight_ele.pt, axis=1) == 1) & (ak.count(tight_mu.pt, axis=1) == 0)
    one_mu = (ak.count(tight_ele.pt, axis=1) == 0) & (ak.count(tight_mu.pt, axis=1) == 1)
    lep_pt = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.pt), ak.where(one_mu, ak.firsts(tight_mu.pt), ak.full_like(met_pt, 0.0))), 0.0)
    lep_phi = ak.fill_none(ak.where(one_ele, ak.firsts(tight_ele.phi), ak.where(one_mu, ak.firsts(tight_mu.phi), ak.full_like(met_pt, 0.0))), 0.0)
    return recoil_pt(met_pt, met_phi, lep_pt * np.cos(lep_phi), lep_pt * np.sin(lep_phi))

fig, ax = plt.subplots()
for name in list(events_by_dataset.keys())[:5]:
    ev = events_by_dataset[name]
    mask = top_cr_mask(ev)
    recoil = top_cr_recoil(ev)[mask]
    if len(ak.ravel(recoil)) == 0:
        continue
    if is_data.get(name, False):
        w = None
    else:
        norm = norm_factors.get(name)
        if norm is None:
            continue
        w = np.full_like(ak.to_numpy(ak.ravel(recoil)), norm, dtype=float)
    ax.hist(ak.to_numpy(ak.ravel(recoil)), bins=30, range=(250, 600), weights=w, label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax.set_xlabel("Recoil [GeV]")
ax.set_ylabel("Events")
ax.set_title("Top CR (single-lepton): Recoil (data and MC)")
ax.legend()
plt.show()


In [ ]:
# cos(theta*) in Top CR (data and MC)
fig2, ax2 = plt.subplots()
for name in list(events_by_dataset.keys()):
    ev = events_by_dataset[name]
    mask = top_cr_mask(ev)
    good_jets_cr = select_good_jets(ev)[mask]
    has_two = ak.num(good_jets_cr) >= 2
    jets_pad = ak.pad_none(good_jets_cr, 2)
    jet0 = jets_pad[:, 0]
    jet1 = jets_pad[:, 1]
    mask2 = has_two & ~ak.is_none(jet1)
    if ak.sum(mask2) == 0:
        continue
    cts = cos_theta_star(jet0[mask2], jet1[mask2])
    flat = ak.to_numpy(ak.ravel(cts))
    if is_data.get(name, False):
        w = None
    else:
        norm = norm_factors.get(name)
        if norm is None:
            continue
        w = np.full_like(flat, norm, dtype=float)
    ax2.hist(flat, bins=25, range=(0, 1), weights=w, label=latex_labels.get(name, labels.get(name, name)), alpha=0.6, histtype="step")
ax2.set_xlabel("$\cos(\Theta)$*")
ax2.set_ylabel("Events")
ax2.set_title("Top CR (single-lepton): $\cos(\Theta)$* (data and MC)")
ax2.legend()
plt.show()

## 9. Comparison plots (Ex 3.4)

In [ ]:
# Use first MC background for comparison plot
name = bkg_names[0] if bkg_names else list(events_by_dataset.keys())[0]
ev = events_by_dataset[name]
good_jets = select_good_jets(ev)
njets = ak.num(good_jets)
nlep = n_tight_leptons(ev)
met = ev.MET.pt

jets2 = ak.pad_none(good_jets, 2)
lead2_b = ak.fill_none(jets2[:, 0].btagDeepFlavB > 0.2783, False) & ak.fill_none(jets2[:, 1].btagDeepFlavB > 0.2783, False)

sr_mask = (njets >= 2) & (njets <= 3) & lead2_b & (nlep == 0) & (met > RECOIL_MIN)
met_presel = met[(njets >= 2) & (njets <= 3) & lead2_b]
recoil = met[sr_mask]

fig, ax = plt.subplots(1, 2, figsize=(15, 8))
ax[0].hist(ak.to_numpy(ak.ravel(met_presel)), bins=50, range=(0, 500), edgecolor="black", alpha=0.7, label="Presel")
ax[0].set_xlabel("MET [GeV]")
ax[0].set_ylabel("Events")
ax[0].set_title("MET (≥1 jet)")
ax[1].hist(ak.to_numpy(ak.ravel(recoil)), bins=50, range=(200, 600), edgecolor="black", alpha=0.7, label="SR")
ax[1].set_xlabel("MET [GeV]")
ax[1].set_ylabel("Events")
ax[1].set_title("MET (signal region)")
plt.tight_layout()
plt.show()